# Multi-Source OOD Computation — Density-Based Typical Set Reference

This notebook computes/caches summary diagnostics using the density-based typical-set detector from `density_based.ipynb`. For each assumed model and summary dimension, training simulations are passed through the summary network, a normalizing-flow density estimator `q_phi(z)` is trained on the latent summaries, and a separate calibration simulation estimates the central 90% typical interval. A third held-out simulator sample, not used for training or reference-distance calibration, validates the learned density by comparing simulator summaries against flow-generated summaries using MMD and C2ST.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_DIR = Path("/Users/yimingzang/Documents/Project/benchmark2")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from benchmark.examples.diffusion.config import TrainingConfig
from benchmark.examples.diffusion.results.multisource_pipeline import (
    all_observed_paths,
    compute_or_load_all_observed,
    reference_path,
)
from benchmark.examples.diffusion.results.observed_datasets import OBSERVED_DATASETS
from benchmark.examples.diffusion.results.summary_diagnostic import MODELS, load_references

INFO:jax._src.xla_bridge:Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/opt/anaconda3/envs/benchmark2/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file), '/usr/local/lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache)
INFO:bayesflow:Using backend 'jax'


In [2]:
from tqdm.auto import tqdm as original_tqdm
import bayesflow.approximators.helpers.samplers as bf_samplers
import bayesflow.approximators.helpers.conditions as bf_conditions


def quiet_tqdm(*args, **kwargs):
    kwargs["disable"] = True
    return original_tqdm(*args, **kwargs)


bf_samplers.tqdm = quiet_tqdm
bf_conditions.tqdm = quiet_tqdm


## Configuration


In [3]:
summary_configs = {
    "S=D": TrainingConfig(summary_multiplier=1),
    "S=2D": TrainingConfig(summary_multiplier=2),
    "S=4D": TrainingConfig(summary_multiplier=4),
    "S=6D": TrainingConfig(summary_multiplier=6),
}

metric = "typical"
num_samples = 2048
mmd_samples = 512
batch_size = 8
recompute = False
recompute_references = True

OBSERVED_DATASETS
# First run: keep recompute_references=True to train/cache *_typical density references.
# Later runs can set recompute_references=False to reuse them.
# Typical references train one density flow per summary dimension × assumed model.

('empirical',
 'simulated_from_m0',
 'simulated_from_m1',
 'simulated_from_m2',
 'simulated_from_m3',
 'm3_fast_30',
 'm3_slow_30',
 'm3_fast_slow_30')

## Compute Or Load Cached Results


In [4]:
diagnostics_by_summary = {}

for label, config in summary_configs.items():
    print(f"Computing/loading {label} ({config.summary_label})")
    diagnostics_by_summary[label] = compute_or_load_all_observed(
        config=config,
        metric=metric,
        num_samples=num_samples,
        mmd_samples=mmd_samples,
        batch_size=batch_size,
        recompute=recompute,
        recompute_references=recompute_references,
    )


Computing/loading S=D (S1D)


INFO:bayesflow:Fitting on dataset instance of OfflineDataset.
INFO:bayesflow:Building on a test batch.


Epoch 1/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 168ms/step - loss: 5.4709
Epoch 2/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 5.3128
Epoch 3/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 5.3046
Epoch 4/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 5.2777
Epoch 5/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 5.2591
Epoch 6/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 5.2425
Epoch 7/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 5.2467
Epoch 8/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 5.2528
Epoch 9/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 5.2115
Epoch 10/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 5.2305
Epoch 11/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 5.2094
Epoch 12/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 5.2041
Epoch 13/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 5.2136
Epoch 14/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 5.2081
Epoch 15/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - l

INFO:bayesflow:Fitting on dataset instance of OfflineDataset.
INFO:bayesflow:Building on a test batch.


Epoch 1/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 147ms/step - loss: 6.6654
Epoch 2/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.4476
Epoch 3/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.3472
Epoch 4/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.3201
Epoch 5/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.3269
Epoch 6/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 6.2729
Epoch 7/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 6.2449
Epoch 8/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.2221
Epoch 9/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.2160
Epoch 10/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.2168
Epoch 11/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.1833
Epoch 12/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.1837
Epoch 13/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 6.2063
Epoch 14/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 6.1773
Epoch 15/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - l

...

Epoch 1/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 169ms/step - loss: 16.9638
Epoch 2/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -3.1657
Epoch 3/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -9.1194
Epoch 4/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: -12.3305
Epoch 5/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -13.7404
Epoch 6/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: -15.1690
Epoch 7/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -16.5298
Epoch 8/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: -17.3369
Epoch 9/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: -17.5497
Epoch 10/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -18.2758
Epoch 11/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: -19.4093
Epoch 12/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: -18.9213
Epoch 13/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -19.6659
Epoch 14/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: -20.2310
Epoch 15/250
16/16 ━━━━━━━━━━━━

INFO:bayesflow:Fitting on dataset instance of OfflineDataset.
INFO:bayesflow:Building on a test batch.


Epoch 1/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 160ms/step - loss: 16.6738
Epoch 2/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -4.4922
Epoch 3/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -10.3737
Epoch 4/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -13.7632
Epoch 5/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -15.7642
Epoch 6/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -17.5768
Epoch 7/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -18.1430
Epoch 8/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -19.2444
Epoch 9/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -19.9084
Epoch 10/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -20.1621
Epoch 11/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -20.9459
Epoch 12/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -21.8480
Epoch 13/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -22.8130
Epoch 14/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -23.0355
Epoch 15/250
16/16 ━━━━━━━━━━━

INFO:bayesflow:Fitting on dataset instance of OfflineDataset.
INFO:bayesflow:Building on a test batch.


Epoch 1/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 182ms/step - loss: 29.0889
Epoch 2/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -0.4452
Epoch 3/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -8.4805
Epoch 4/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -13.3541
Epoch 5/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: -16.2766
Epoch 6/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -17.8592
Epoch 7/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -19.5600
Epoch 8/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -20.9884
Epoch 9/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -22.1830
Epoch 10/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: -22.8730
Epoch 11/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: -23.9000
Epoch 12/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -24.2431
Epoch 13/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -24.7670
Epoch 14/250
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: -25.6708
Epoch 15/250
16/16 ━━━━━━━━━━━━

## Cached Files


In [5]:
cache_rows = []
for label, config in summary_configs.items():
    for name, path in all_observed_paths(config.summary_label, metric=metric).items():
        cache_rows.append(
            {
                "summary": label,
                "file_type": name,
                "path": str(path),
                "exists": path.exists(),
            }
        )

cache_files = pd.DataFrame(cache_rows)
cache_files


,summary,file_type,path,exists
0,S=D,results,/Users/yimingzang/Documents/Project/benchmark2...,True
1,S=D,diagnostic,/Users/yimingzang/Documents/Project/benchmark2...,True
2,S=D,posterior,/Users/yimingzang/Documents/Project/benchmark2...,True
3,S=D,posterior_plot,/Users/yimingzang/Documents/Project/benchmark2...,True
4,S=2D,results,/Users/yimingzang/Documents/Project/benchmark2...,True
5,S=2D,diagnostic,/Users/yimingzang/Documents/Project/benchmark2...,True
6,S=2D,posterior,/Users/yimingzang/Documents/Project/benchmark2...,True
7,S=2D,posterior_plot,/Users/yimingzang/Documents/Project/benchmark2...,True
8,S=4D,results,/Users/yimingzang/Documents/Project/benchmark2...,True
9,S=4D,diagnostic,/Users/yimingzang/Documents/Project/benchmark2...,True


## Sanity Checks


In [6]:
coverage_rows = []
for label, frames in diagnostics_by_summary.items():
    for name, frame in frames.items():
        coverage_rows.append(
            {
                "summary": label,
                "frame": name,
                "rows": len(frame),
                "datasets": ", ".join(frame["dataset"].drop_duplicates()),
                "n_datasets": frame["dataset"].nunique(),
            }
        )

coverage = pd.DataFrame(coverage_rows)
coverage


,summary,frame,rows,datasets,n_datasets
0,S=D,results,136,"empirical, simulated_from_m0, simulated_from_m...",8
1,S=D,diagnostic,136,"empirical, simulated_from_m0, simulated_from_m...",8
2,S=D,posterior,544,"empirical, simulated_from_m0, simulated_from_m...",8
3,S=D,posterior_plot,544,"empirical, simulated_from_m0, simulated_from_m...",8
4,S=2D,results,136,"empirical, simulated_from_m0, simulated_from_m...",8
5,S=2D,diagnostic,136,"empirical, simulated_from_m0, simulated_from_m...",8
6,S=2D,posterior,544,"empirical, simulated_from_m0, simulated_from_m...",8
7,S=2D,posterior_plot,544,"empirical, simulated_from_m0, simulated_from_m...",8
8,S=4D,results,136,"empirical, simulated_from_m0, simulated_from_m...",8
9,S=4D,diagnostic,136,"empirical, simulated_from_m0, simulated_from_m...",8


In [7]:
posterior_summary = []
for label, frames in diagnostics_by_summary.items():
    frame = frames["posterior"]
    posterior_summary.append(
        frame.groupby(["dataset", "model"], sort=False)
        .agg(
            mean_mmd=("posterior_mmd", "mean"),
            median_mmd=("posterior_mmd", "median"),
            mean_posterior_mean_rmse=("posterior_mean_rmse", "mean"),
        )
        .assign(summary_dimension=label)
        .reset_index()
    )

posterior_summary = pd.concat(posterior_summary, ignore_index=True)
posterior_summary


,dataset,model,mean_mmd,median_mmd,mean_posterior_mean_rmse,summary_dimension
0,empirical,m0,0.219341,0.154009,0.238050,S=D
1,empirical,m1,0.227051,0.175034,0.194805,S=D
2,empirical,m2,0.211853,0.221962,0.220987,S=D
3,empirical,m3,0.306619,0.212100,0.259090,S=D
4,simulated_from_m0,m0,0.010768,0.008996,0.040133,S=D
...,...,...,...,...,...,...
123,m3_slow_30,m3,0.065447,0.056301,0.102677,S=6D
124,m3_fast_slow_30,m0,0.281155,0.179708,0.416993,S=6D
125,m3_fast_slow_30,m1,0.442371,0.418471,0.477028,S=6D
126,m3_fast_slow_30,m2,0.095229,0.064474,0.140913,S=6D


## Density Flow Held-Out Validation\n
\n
For each summary dimension and assumed model, compare held-out simulator summaries against samples generated from the learned density flow. C2ST accuracy close to 0.5 indicates that the classifier cannot reliably distinguish the two samples; MMD should also be small relative to other fits.\n

In [8]:
validation_rows = []
for label, config in summary_configs.items():
    references = load_references(reference_path(config.summary_label, metric=metric))
    for model in MODELS:
        ref = references[model]
        validation_rows.append(
            {
                "summary_dimension": label,
                "model": model,
                "c2st_accuracy": ref.get("density_validation_c2st_accuracy"),
                "mmd": ref.get("density_validation_mmd"),
                "mmd2": ref.get("density_validation_mmd2"),
                "n_simulator": ref.get("density_validation_n_simulator"),
                "n_flow": ref.get("density_validation_n_flow"),
                "typicality_low": ref.get("typicality_low"),
                "typicality_high": ref.get("typicality_high"),
            }
        )

density_flow_validation = pd.DataFrame(validation_rows)
density_flow_validation

,summary_dimension,model,c2st_accuracy,mmd,mmd2,n_simulator,n_flow,typicality_low,typicality_high
0,S=D,m0,0.5135,0.049187,0.002419,2000,2000,-3.899140,2.380388
1,S=D,m1,0.5260,0.047284,0.002236,2000,2000,-4.559561,2.515721
2,S=D,m2,0.5040,0.054664,0.002988,2000,2000,-4.696848,2.605484
3,S=D,m3,0.5130,0.034037,0.001159,2000,2000,-9.360566,4.760859
4,S=2D,m0,0.5060,0.038967,0.001518,2000,2000,-6.096990,3.978201
5,S=2D,m1,0.5075,0.043318,0.001876,2000,2000,-8.099177,5.115617
6,S=2D,m2,0.5030,0.033492,0.001122,2000,2000,-7.577586,4.547107
7,S=2D,m3,0.5315,0.050568,0.002557,2000,2000,-10.982581,7.347402
8,S=4D,m0,0.5145,0.029674,0.000881,2000,2000,-10.362528,6.971862
9,S=4D,m1,0.4935,0.038282,0.001465,2000,2000,-14.912279,10.095640
